# Daniel OS grounded SFT: Colab GPU experiment

This notebook audits the old loss curve, generates diverse prompts while keeping every answer curated, profiles the complete text-training input path, benchmarks DataLoader and compiler settings, builds scenario-family-disjoint train/validation splits, and compares LFM2 with LFM2.5 across three learning rates. Publication remains disabled until the frozen strict-test baseline is met or exceeded.

The data design follows the quality-over-volume findings in [LIMA](https://papers.neurips.cc/paper_files/paper/2023/hash/ac662d74829e4407ce1d126477f4a03a-Abstract-Conference.html), the generate-and-filter pattern in [Self-Instruct](https://aclanthology.org/2023.acl-long.754/), and the quality/complexity/diversity framework in [DEITA](https://proceedings.iclr.cc/paper_files/paper/2024/hash/6091f2bb355e960600f62566ac0e2862-Abstract-Conference.html).

In [ ]:
import os, pathlib, subprocess, sys

REPO = pathlib.Path('/content/sangbumchoi.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SangbumChoi/sangbumchoi.github.io.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch>=2.6', 'transformers>=4.55,<5', 'trl>=0.24,<0.30', 'peft>=0.17,<1', 'datasets>=3,<5', 'accelerate>=1.2,<2', 'bitsandbytes>=0.45,<1', 'huggingface-hub>=0.34,<2', 'matplotlib>=3.9'], check=True)
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=True).stdout)

## 1. Diagnose the previous run

The old loss holdout had only two records per behavior. The audit also reconstructs how much cyclic oversampling occurred and checks the frozen evaluations for exact or near prompt overlap.

In [ ]:
subprocess.run([sys.executable, 'scripts/analyze_daniel_lfm2_data.py'], check=True)
diagnostics = __import__('json').loads(pathlib.Path('artifacts/daniel-lfm2-data-diagnostics.json').read_text())
print(__import__('json').dumps({
    'legacy_sampling': diagnostics['legacy_sampling'],
    'loss': diagnostics['loss'],
    'near_prompt_matches': diagnostics['prompt_overlap']['near_prompt_matches'],
}, indent=2))

## 2. Refresh the source survey

This verifies source availability, hashes retrieved pages, and creates review excerpts. Network text is never promoted directly into SFT. A human must update the curated profile before a new fact can become a target.

In [ ]:
subprocess.run([sys.executable, 'scripts/survey_daniel_public_sources.py'], check=True)
survey = __import__('json').loads(pathlib.Path('artifacts/daniel-public-source-survey.json').read_text())
survey['summary']

## 3. Generate grounded prompt diversity

A 4-bit Qwen3-4B teacher writes prompts only. The canonical answer, context keys, evidence, behavior, and language come from curated seeds. Entity definitions are expanded from the cited entity index, and neutral world-knowledge questions teach a search tool call rather than model-memory answers.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/generate_daniel_lfm2_synthetic.py',
    '--output-dir', 'artifacts/daniel-lfm2-v3',
    '--batch-size', '4',
    '--temperature', '0.7',
], check=True)
subprocess.run([
    sys.executable, 'scripts/validate_daniel_lfm2_data.py',
    '--prepared-train', 'artifacts/daniel-lfm2-v3/train.jsonl',
    '--prepared-validation', 'artifacts/daniel-lfm2-v3/validation.jsonl',
], check=True)

## 4. Profile before optimizing

The [Trillion Labs VLM speed study](https://blog.trillionlabs.co/posts/vlm-training-speed/) treats throughput as a pipeline problem rather than a model-only problem. For this text-only SFT, image decoding and resolution variance become chat-template tokenization, dynamic padding, collation, host-to-device transfer, and sequence-length variance. The cells below therefore measure the exact Daniel OS data path before choosing an optimization.

The diagnostic tools follow the official [PyTorch Profiler](https://docs.pytorch.org/docs/stable/profiler.html), [PyTorch data-loading optimization](https://docs.pytorch.org/tutorials/intermediate/intermediate_data_loading_tutorial.html), and [Transformers `torch.compile`](https://huggingface.co/docs/transformers/torch_compile) guidance. A profiler trace has deliberate overhead, so it is never used as the throughput benchmark.

In [ ]:
import gc, importlib.util, json, random, statistics, torch
from transformers import AutoTokenizer

spec = importlib.util.spec_from_file_location('daniel_train', 'scripts/train_daniel_lfm2.py')
daniel_train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(daniel_train)

profile_data = json.loads(pathlib.Path('assets/data/daniel-profile.json').read_text())
prepared_train = [json.loads(line) for line in pathlib.Path('artifacts/daniel-lfm2-v3/train.jsonl').read_text().splitlines() if line.strip()]
speed_model = 'LiquidAI/LFM2-350M'
speed_tokenizer = AutoTokenizer.from_pretrained(speed_model)

token_lengths = []
for record in prepared_train:
    conversation = [
        {'role': 'system', 'content': daniel_train.system_prompt(profile_data, record['context_keys'], record.get('evidence'))},
        *record['messages'],
    ]
    token_lengths.append(len(speed_tokenizer.apply_chat_template(conversation, tokenize=True)))

def padding_efficiency(lengths, batch_size=8, group_by_length=False):
    ordered = list(lengths)
    if group_by_length:
        ordered.sort()
    else:
        random.Random(42).shuffle(ordered)
    padded = 0
    for start in range(0, len(ordered), batch_size):
        batch = ordered[start:start + batch_size]
        padded += max(batch) * len(batch)
    return sum(ordered) / padded

quantiles = statistics.quantiles(token_lengths, n=100, method='inclusive')
length_report = {
    'records': len(token_lengths),
    'minimum': min(token_lengths),
    'median': statistics.median(token_lengths),
    'p95': quantiles[94],
    'p99': quantiles[98],
    'maximum': max(token_lengths),
    'over_max_length_1152': sum(length > 1152 for length in token_lengths),
    'random_batch_padding_efficiency': round(padding_efficiency(token_lengths), 4),
    'length_grouped_upper_bound': round(padding_efficiency(token_lengths, group_by_length=True), 4),
}
print(json.dumps(length_report, indent=2))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(token_lengths, bins=30, color='#168b6b', edgecolor='white')
ax.axvline(1152, color='#c9463d', linestyle='--', label='max_length=1152')
ax.set(xlabel='tokens after exact chat template', ylabel='records', title='Daniel OS SFT sequence-length distribution')
ax.grid(axis='y', alpha=.2); ax.legend(); fig.tight_layout(); plt.show()

assert length_report['over_max_length_1152'] == 0, 'Increase max_length; a completion would be truncated.'

### 4.1 Capture one bounded `torch.profiler` trace

The profiler waits for two optimizer steps, warms up for two, and records five. It writes a Chrome/Perfetto trace, an operator table, and a machine-readable top-operation summary. CPU events reveal collation and launch gaps; CUDA events reveal attention, matrix multiplication, copies, and memory allocation. DataLoader subprocess work is not always fully visible in the parent trace, so the worker sweep in the next section is the end-to-end test for an input bottleneck.

In [ ]:
speed_root = pathlib.Path('artifacts/daniel-lfm2-speed')
speed_root.mkdir(parents=True, exist_ok=True)
FORCE_SPEED_RERUN = False

def run_speed_trial(name, workers=0, max_steps=30, compile_model=False, profile_trace=False):
    output = speed_root / name
    profile_dir = output / 'profiler'
    required = [output / 'training_metrics.json', output / 'performance_config.json', output / 'dataloader_metrics.json']
    profile_ready = not profile_trace or any(profile_dir.glob('summary-step-*.json'))
    if FORCE_SPEED_RERUN or not all(path.exists() for path in required) or not profile_ready:
        command = [
            sys.executable, 'scripts/train_daniel_lfm2.py',
            '--model', speed_model,
            '--prepared-train', 'artifacts/daniel-lfm2-v3/train.jsonl',
            '--prepared-validation', 'artifacts/daniel-lfm2-v3/validation.jsonl',
            '--output', str(output), '--benchmark-only',
            '--max-steps', str(max_steps), '--max-length', '1152',
            '--batch-size', '8', '--eval-batch-size', '8',
            '--gradient-accumulation-steps', '4',
            '--dataloader-num-workers', str(workers),
            '--dataloader-benchmark-batches', '20',
            '--include-tokens-per-second',
            '--training-revision', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
        ]
        if workers > 0:
            command += ['--dataloader-prefetch-factor', '2', '--dataloader-persistent-workers']
        if compile_model:
            command += ['--torch-compile', '--torch-compile-mode', 'default']
        if profile_trace:
            command += [
                '--profile-output', str(profile_dir),
                '--profile-wait-steps', '2', '--profile-warmup-steps', '2', '--profile-active-steps', '5',
            ]
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        (speed_root / f'{name}.log').write_text(result.stdout)
        if result.returncode:
            print('\n'.join(result.stdout.splitlines()[-80:]))
            result.check_returncode()
    return output

gc.collect(); torch.cuda.empty_cache()
profile_run = run_speed_trial('profile_workers0', workers=0, max_steps=12, profile_trace=True)
profile_summaries = sorted((profile_run / 'profiler').glob('summary-step-*.json'))
assert profile_summaries, 'Profiler schedule did not complete; increase --max-steps.'
profile_summary = json.loads(profile_summaries[-1].read_text())
for operation in profile_summary['top_operations'][:15]:
    print(f"{operation['name'][:70]:70s} CPU {operation['self_cpu_time_us']/1000:9.2f} ms  device {operation['self_device_time_us']/1000:9.2f} ms")
print('Trace:', profile_summary['trace'])
print('Open the trace in https://ui.perfetto.dev/ or download it from the Colab file browser.')

### 4.2 Benchmark the input pipeline without profiler overhead

Each trial uses the same model, seed, batch size, gradient accumulation, data, and optimizer-step count. `pin_memory=True` remains enabled in every CUDA trial. Worker trials add asynchronous loading, two prefetched batches per worker, and persistent workers. Select a worker count only when the measured gain is at least 3%; otherwise keep zero workers and save Colab RAM/process overhead.

Interpretation guide:

- **Workers improve tokens/sec materially:** input collation/loading was starving the trainer; carry the best worker count into the full sweep.
- **Workers do not help:** this in-memory text dataset is compute- or launch-bound; more workers only add overhead.
- **Low padding efficiency:** retain `group_by_length=True`; consider packing only in a separate quality-equivalence experiment because it changes sample boundaries.
- **Long CPU gaps and many tiny CUDA launches:** test `torch.compile` or a larger microbatch. Compilation has a slow first step, so compare a sufficiently long unprofiled run.
- **Out of memory:** lower the microbatch or enable gradient checkpointing. Checkpointing saves memory by recomputation and normally reduces speed, so it is not a speed optimization by itself.

In [ ]:
worker_rows = []
for workers in (0, 2, 4):
    gc.collect(); torch.cuda.empty_cache()
    output = run_speed_trial(f'workers{workers}', workers=workers, max_steps=30)
    metrics = json.loads((output / 'training_metrics.json').read_text())['summary']
    config = json.loads((output / 'performance_config.json').read_text())
    loader = json.loads((output / 'dataloader_metrics.json').read_text())
    worker_rows.append({
        'workers': workers,
        'tokens_per_second': metrics.get('train_tokens_per_second'),
        'samples_per_second': metrics.get('train_samples_per_second'),
        'steps_per_second': metrics.get('train_steps_per_second'),
        'runtime_seconds': metrics.get('train_runtime_seconds'),
        'pin_memory': config['dataloader']['pin_memory'],
        'persistent_workers': config['dataloader']['persistent_workers'],
        'first_batch_seconds': loader['first_batch_seconds'],
        'fetch_p50_seconds': loader['steady_fetch_p50_seconds'],
        'fetch_p95_seconds': loader['steady_fetch_p95_seconds'],
        'loader_only_samples_per_second': loader['loader_only_samples_per_second'],
    })

metric_name = 'tokens_per_second' if all(row['tokens_per_second'] for row in worker_rows) else 'samples_per_second'
baseline_rate = next(row[metric_name] for row in worker_rows if row['workers'] == 0)
best_row = max(worker_rows, key=lambda row: row[metric_name])
measured_speedup = best_row[metric_name] / baseline_rate
BEST_DATALOADER_WORKERS = best_row['workers'] if measured_speedup >= 1.03 else 0
BEST_DATALOADER_PREFETCH = 2
print(json.dumps({'metric': metric_name, 'trials': worker_rows, 'best_measured_speedup': measured_speedup, 'selected_workers': BEST_DATALOADER_WORKERS}, indent=2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(row['workers']) for row in worker_rows], [row[metric_name] for row in worker_rows], color=['#6d7378', '#168b6b', '#1f6f9c'])
ax.set(xlabel='DataLoader workers', ylabel=metric_name.replace('_', ' '), title='Unprofiled input-pipeline benchmark')
ax.grid(axis='y', alpha=.2); fig.tight_layout(); plt.show()

# Compilation is opt-in because its first-step cost can dominate a short Colab run.
RUN_COMPILE_ABLATION = False
USE_TORCH_COMPILE = False
if RUN_COMPILE_ABLATION:
    eager_output = run_speed_trial('eager60', workers=BEST_DATALOADER_WORKERS, max_steps=60)
    compile_output = run_speed_trial('compile60', workers=BEST_DATALOADER_WORKERS, max_steps=60, compile_model=True)
    eager_metrics = json.loads((eager_output / 'training_metrics.json').read_text())['summary']
    compile_metrics = json.loads((compile_output / 'training_metrics.json').read_text())['summary']
    compile_rate = compile_metrics.get('train_tokens_per_second') or compile_metrics['train_samples_per_second']
    eager_rate = eager_metrics.get('train_tokens_per_second') or eager_metrics['train_samples_per_second']
    USE_TORCH_COMPILE = compile_rate / eager_rate >= 1.05
    print({'compile_rate': compile_rate, 'eager_rate': eager_rate, 'use_in_full_sweep': USE_TORCH_COMPILE})

if BEST_DATALOADER_WORKERS > 0:
    bottleneck = 'input_pipeline'
    reason = f'Workers improved {metric_name} by {(measured_speedup - 1) * 100:.1f}%.'
elif length_report['length_grouped_upper_bound'] < 0.85:
    bottleneck = 'sequence_padding'
    reason = 'Even ideal length grouping leaves more than 15% padding waste.'
else:
    bottleneck = 'gpu_compute_or_launch_overhead'
    reason = 'Extra DataLoader workers did not clear the 3% threshold and grouped padding is acceptable.'
diagnosis = {
    'bottleneck': bottleneck,
    'reason': reason,
    'selected_workers': BEST_DATALOADER_WORKERS,
    'worker_speedup': measured_speedup,
    'padding_efficiency': length_report['length_grouped_upper_bound'],
    'selected_loader_fetch_p95_seconds': next(row['fetch_p95_seconds'] for row in worker_rows if row['workers'] == BEST_DATALOADER_WORKERS),
    'top_device_operations': [operation['name'] for operation in profile_summary['top_operations'][:10]],
    'next_test': 'Run the equal-length eager/compile ablation.' if bottleneck == 'gpu_compute_or_launch_overhead' else 'Carry the measured input setting into the full sweep.',
}
(speed_root / 'speed-diagnosis.json').write_text(json.dumps(diagnosis, indent=2))
print(json.dumps(diagnosis, indent=2))

## 5. GPU ablation sweep

The sweep uses an equal-size per-behavior validation pool for checkpoint selection and also logs full per-behavior loss. Evaluation runs every 25 optimizer steps, with two-evaluation early stopping. The frozen behavioral and strict suites are run for every candidate. The measured DataLoader choice from the speed lab is reused; `torch.compile` remains off unless its separate, longer ablation wins by at least 5%.

In [ ]:
import gc, json, torch
gc.collect(); torch.cuda.empty_cache()

BEST_DATALOADER_WORKERS = globals().get('BEST_DATALOADER_WORKERS', 0)
BEST_DATALOADER_PREFETCH = globals().get('BEST_DATALOADER_PREFETCH', 2)
USE_TORCH_COMPILE = globals().get('USE_TORCH_COMPILE', False)

models = ['LiquidAI/LFM2-350M', 'LiquidAI/LFM2.5-350M']
learning_rates = [5e-5, 1e-4, 2e-4]
sweep_root = pathlib.Path('artifacts/daniel-lfm2-sweep')
sweep_root.mkdir(parents=True, exist_ok=True)

for model in models:
    for lr in learning_rates:
        run_name = f"{model.split('/')[-1]}_lr{lr:.0e}"
        output = sweep_root / run_name
        command = [
            sys.executable, 'scripts/train_daniel_lfm2.py',
            '--model', model,
            '--prepared-train', 'artifacts/daniel-lfm2-v3/train.jsonl',
            '--prepared-validation', 'artifacts/daniel-lfm2-v3/validation.jsonl',
            '--output', str(output),
            '--epochs', '3', '--max-length', '1152',
            '--batch-size', '8', '--eval-batch-size', '8',
            '--gradient-accumulation-steps', '4',
            '--learning-rate', str(lr), '--weight-decay', '0.01',
            '--eval-steps', '25', '--early-stopping-patience', '2',
            '--dataloader-num-workers', str(BEST_DATALOADER_WORKERS),
            '--training-revision', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
        ]
        if BEST_DATALOADER_WORKERS > 0:
            command += ['--dataloader-prefetch-factor', str(BEST_DATALOADER_PREFETCH), '--dataloader-persistent-workers']
        if USE_TORCH_COMPILE:
            command += ['--torch-compile', '--torch-compile-mode', 'default']
        with (output.parent / f'{run_name}.log').open('w') as log:
            subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, check=True)
        subprocess.run([
            sys.executable, 'scripts/evaluate_daniel_lfm2_test.py',
            '--model', str(output / 'merged'),
            '--output', str(output / 'strict-evaluation.json'),
            '--minimum-strict-score', '0.0', '--minimum-answer-score', '0.0',
            '--minimum-retrieve-score', '0.0', '--minimum-unknown-score', '0.0',
            '--minimum-refuse-score', '0.0', '--minimum-korean-score', '0.0',
        ], check=True)
        gc.collect(); torch.cuda.empty_cache()
        print('finished', run_name)

## 6. Select, plot, and export the result

A run is publishable only if strict, macro, answer, unknown, retrieval, refusal, hallucination, and Korean gates are all at least as strong as the currently deployed checkpoint.

In [ ]:
subprocess.run([sys.executable, 'scripts/select_daniel_lfm2_candidate.py', str(sweep_root)], check=True)
selection = json.loads((sweep_root / 'selection.json').read_text())
print(json.dumps(selection, indent=2))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
for run in selection['runs']:
    metrics = json.loads((pathlib.Path(run['path']) / 'training_metrics.json').read_text())
    points = [p for p in metrics['validation'] if p.get('dataset') == 'macro' and p.get('loss') is not None]
    ax.plot([p['step'] for p in points], [p['loss'] for p in points], marker='o', label=run['run'])
ax.set(xlabel='optimizer step', ylabel='balanced validation loss', title='Daniel OS LFM2 ablation')
ax.grid(alpha=.25); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(sweep_root / 'validation-loss-ablation.png', dpi=180)
plt.show()

## 7. Quantize and replay the same questions

Quantization is not accepted from file size or a one-prompt smoke test. This stage first replays the frozen strict set against the selected source checkpoint on CPU, exports symmetric Q4 ONNX, and then replays the exact serialized messages with the same greedy decoding settings through ONNX Runtime on the same CPU. The report lists every changed answer, strict-score change, retained key facts, new forbidden-information leaks, tokenization parity, median latency, and token throughput. Wording may change without failing the gate, but factual or privacy regressions do block publication. CPU parity isolates quantization/runtime effects; browser-specific WebGPU speed remains a separate hardware benchmark.

In [ ]:
RUN_Q4_PARITY = True  # Required for release; set False only during exploratory sweeps.
selected_run = pathlib.Path(selection['selected']['path'])
selected = selected_run / 'merged'
q4_root = selected_run / 'q4-parity'
source_cpu_evaluation = q4_root / 'source-cpu-evaluation.json'
quantization_parity_path = q4_root / 'quantization-parity.json'

if RUN_Q4_PARITY:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    q4_root.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, 'scripts/evaluate_daniel_lfm2_test.py',
        '--model', str(selected), '--device', 'cpu',
        '--output', str(source_cpu_evaluation),
    ], check=True)
    subprocess.run([
        'uv', 'run', 'scripts/export_daniel_lfm2_onnx.py',
        '--source', str(selected), '--output-dir', str(q4_root),
        '--baseline-evaluation', str(source_cpu_evaluation),
    ], check=True)

quantization_parity = json.loads(quantization_parity_path.read_text()) if quantization_parity_path.exists() else None
if quantization_parity:
    print(json.dumps({
        'publication_allowed': quantization_parity['publication_allowed'],
        'quality': quantization_parity['quality'],
        'performance': quantization_parity['performance'],
        'changed_answer_count': len(quantization_parity['changed_answers']),
    }, indent=2))
else:
    print('Q4 parity has not run; publication remains blocked.')

In [ ]:
# Deliberate release switch. Keep False until the selection report is reviewed.
PUBLISH = False
if PUBLISH:
    assert selection['publication_allowed'], 'No candidate passed every frozen baseline gate.'
    assert quantization_parity and quantization_parity['publication_allowed'], 'Q4 parity gate has not passed.'
    from google.colab import userdata
    from huggingface_hub import HfApi
    token = userdata.get('HF_TOKEN')
    assert token, 'Add HF_TOKEN to Colab Secrets; never paste it into the notebook.'
    api = HfApi(token=token)
    api.create_repo('danelcsb/daniel-lfm2-350m-candidate', repo_type='model', exist_ok=True)
    api.upload_folder(
        repo_id='danelcsb/daniel-lfm2-350m-candidate',
        repo_type='model', folder_path=selected,
        commit_message='Upload strict-gated Daniel OS candidate',
    )
    print('Candidate uploaded. Browser ONNX remains unchanged until its export and smoke tests pass.')